In [ ]:
# =============================================================================
# Zelle 01 – Google Drive mounten
# =============================================================================
# Drive wird eingebunden, damit alle Dateien persistent gespeichert werden.
# Ohne Mount gehen alle Daten bei Session-Ende verloren.

from google.colab import drive

drive.mount('/content/drive')

# Basispfad definieren – EINMAL hier setzen, überall referenzieren
BASE_PATH = '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10'

print(f"Base Path: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base Path: /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10


### Zelle 01 — Google Drive mounten

#### Was
Google Drive wird in das Colab-Dateisystem eingebunden unter `/content/drive`.
Der Basispfad `BASE_PATH` zeigt auf den Projektordner in Drive.

#### Warum
Colab ist eine temporäre Umgebung — bei Session-Ende gehen alle lokalen Daten verloren.
Drive ist persistent: Notebooks, Modelle, Plots und Metriken bleiben erhalten.
Ohne Drive-Mount wäre jede Session ein Neustart ohne Ergebnisse.

#### Konzept: Persistenz in Cloud-Umgebungen
Colab arbeitet auf einem temporären virtuellen Server (Google-Infrastruktur).
Dieser Server wird nach ~90 Min Inaktivität oder ~12h Laufzeit zurückgesetzt.
Drive ist davon unabhängig — es ist externer Speicher, nicht Teil des Servers.
Der Mount-Befehl verbindet beide Systeme über das Dateisystem.

#### Ergebnis
Mounted at /content/drive
Base Path: /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10
Drive erfolgreich eingebunden. Alle nachfolgenden Zellen können auf `BASE_PATH` zugreifen.

In [ ]:
# =============================================================================
# Zelle 02 – Ordnerstruktur anlegen
# =============================================================================
# Alle benötigten Ordner werden einmalig erstellt.
# exist_ok=True verhindert Fehler wenn Ordner bereits existiert.

import os

# Alle Pfade relativ zu BASE_PATH definieren
folders = [
    'notebooks',
    'models',
    'reports/figures',
    'reports/metrics',
    'reports/misclassified',
    'presentation',
]

for folder in folders:
    full_path = os.path.join(BASE_PATH, folder)
    os.makedirs(full_path, exist_ok=True)
    print(f"✓ {full_path}")

print("\nOrdnerstruktur erfolgreich angelegt.")

✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/notebooks
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/models
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/figures
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/metrics
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/reports/misclassified
✓ /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/presentation

Ordnerstruktur erfolgreich angelegt.


### Zelle 02 — Ordnerstruktur anlegen

#### Was
Alle Projektordner werden einmalig auf Google Drive angelegt.
`os.makedirs()` mit `exist_ok=True` erstellt Ordner idempotent —
mehrfaches Ausführen verursacht keinen Fehler.

#### Warum
Eine klare Ordnerstruktur ist Grundvoraussetzung für reproduzierbare,
professionelle Projekte. Jeder Ordner hat eine definierte Verantwortlichkeit:

| Ordner | Inhalt |
|--------|--------|
| `notebooks/` | Jupyter Notebooks pro Phase |
| `models/` | Gespeicherte Modellgewichte (.keras) |
| `reports/figures/` | Plots — benannt nach Notebook und Inhalt |
| `reports/metrics/` | CSV-Dateien mit Messergebnissen |
| `reports/misclassified/` | Falsch klassifizierte Bilder zur Fehleranalyse |
| `presentation/` | Präsentationsfolien (Phase 7) |

#### Konzept: Projektstruktur in ML-Projekten
In produktiven ML-Projekten trennt man bewusst:
- **Rohdaten** (unveränderlich, nur lesen)
- **Code** (versioniert in GitHub)
- **Artefakte** (Modelle, Plots — reproduzierbar aus Code + Daten)

Diese Trennung ermöglicht Reproduzierbarkeit:
Gleicher Code + gleiche Daten = gleiche Artefakte.
Ohne Struktur entstehen "Magic Notebooks" — funktionieren einmal,
niemand (auch man selbst nicht) versteht später warum.

#### Ergebnis
6 Ordner erfolgreich angelegt. Struktur ist persistent auf Drive gespeichert.


In [ ]:
# =============================================================================
# Zelle 03 – GPU prüfen und dokumentieren
# =============================================================================
# Ohne GPU ist Training auf CIFAR-10 mit ResNet50 nicht praktikabel.
# Diese Zelle prüft welche Hardware verfügbar ist und dokumentiert dies.
# Bei "Not connected to a GPU" → Colab Runtime ändern:
# Laufzeit → Laufzeittyp ändern → T4 GPU

import tensorflow as tf

print("=" * 50)
print("HARDWARE CHECK")
print("=" * 50)

# TensorFlow Version
print(f"\nTensorFlow Version : {tf.__version__}")

# GPU verfügbar?
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        print(f"GPU erkannt        : {gpu.name}")

    # GPU Details via nvidia-smi
    print("\nGPU Details:")
    os.system('nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv')
else:
    print("WARNUNG: Keine GPU erkannt.")
    print("→ Laufzeit → Laufzeittyp ändern → T4 GPU auswählen")

# RAM Check
import psutil
ram = psutil.virtual_memory()
print(f"\nRAM total          : {ram.total / 1e9:.1f} GB")
print(f"RAM verfügbar      : {ram.available / 1e9:.1f} GB")

print("\n" + "=" * 50)

HARDWARE CHECK

TensorFlow Version : 2.20.0
GPU erkannt        : /physical_device:GPU:0

GPU Details:

RAM total          : 13.6 GB
RAM verfügbar      : 11.3 GB



### Zelle 03 — GPU prüfen und dokumentieren

#### Was
Verfügbare Hardware wird geprüft und dokumentiert:
- GPU-Typ und VRAM
- TensorFlow-Version
- Verfügbarer RAM

#### Warum
Training eines CNN auf CIFAR-10 ohne GPU ist praktisch nicht durchführbar.
Vergleich:

| Hardware | Trainingszeit pro Epoche (ResNet50, 10k Bilder) |
|----------|------------------------------------------------|
| CPU | ~15–30 Min |
| T4 GPU | ~1–2 Min |
| A100 GPU | ~15–30 Sek |

Faktor ~15x Geschwindigkeit zwischen CPU und T4.
Bei 20 Epochen (Basis-Projekt): CPU ~600 Min vs. T4 ~40 Min.
GPU ist keine Komfortoption — sie ist Voraussetzung für praktikables Training.

#### Konzept: Warum GPU für Deep Learning?
Neuronale Netze bestehen aus Millionen von Matrizenmultiplikationen.
GPU vs. CPU — fundamentaler Architekturunterschied:

| | CPU | GPU |
|-|-----|-----|
| Kerne | 8–32 | 2.000–10.000 |
| Optimiert für | Sequentielle Aufgaben | Parallele Aufgaben |
| Matrixmultiplikation | Langsam | Extrem schnell |

TensorFlow erkennt GPU automatisch und verschiebt Berechnungen dorthin.
`tf.config.list_physical_devices('GPU')` gibt zurück welche GPUs verfügbar sind.

#### Konzept: Colab Free GPU Limits
Colab Free stellt T4 GPU zur Verfügung mit folgenden Einschränkungen:
- ~12 GB VRAM
- ~3–4h kontinuierliche Nutzung pro Session
- Tägliches GPU-Kontingent (bei Überschreitung: nur CPU verfügbar)

**Konsequenz für dieses Projekt:**
Modellgewichte werden nach jeder Epoche gespeichert (ModelCheckpoint).
Training kann bei Disconnect fortgesetzt werden — kein Datenverlust.

#### Ergebnis
TensorFlow Version : 2.20.0
GPU erkannt        : /physical_device:GPU:0
RAM total          : 13.6 GB
T4 GPU verfügbar. Training ist praktikabel.

In [ ]:
# =============================================================================
# Zelle 04 – Zentrale Imports
# =============================================================================
# Alle Standard-Bibliotheken die im gesamten Notebook verwendet werden.
# Colab-spezifische und zellen-lokale Imports bleiben dort wo sie gebraucht werden.

# Standard Library
import os
import random
import warnings
warnings.filterwarnings('ignore')

# Numerik
import numpy as np

# Visualisierung
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras

# Reproduzierbarkeit – WICHTIG
# Alle Zufallsgeneratoren auf denselben Seed setzen damit Ergebnisse reproduzierbar sind
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Globale Konstanten
BASE_PATH = '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10'

print("Imports erfolgreich.")
print(f"NumPy     : {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Seed      : {SEED}")

Imports erfolgreich.
NumPy     : 2.0.2
TensorFlow: 2.20.0
Seed      : 42


### Zelle 04 — Zentrale Imports + Seed

#### Was
Alle Standard-Bibliotheken werden geladen und ein globaler Zufallsseed gesetzt.
Vier Zufallsgeneratoren werden gleichzeitig fixiert:
- Python `random`
- NumPy
- TensorFlow
- Umgebungsvariable `PYTHONHASHSEED`

#### Warum — Bibliotheken
Jede Bibliothek hat eine spezifische Aufgabe:

| Bibliothek | Aufgabe in diesem Projekt |
|------------|--------------------------|
| `os` | Dateisystem, Pfade, Umgebungsvariablen |
| `random` | Python-nativer Zufallsgenerator |
| `numpy` | Numerische Operationen, Array-Manipulation |
| `matplotlib` | Plots, Visualisierungen |
| `seaborn` | Statistische Visualisierungen (Heatmaps, Verteilungen) |
| `tensorflow` | Modellbau, Training, Evaluation |
| `keras` | High-Level API für TensorFlow |

#### Warum — Seed

Ein Seed ist ein Startwert für Zufallsgeneratoren.
Ohne fixen Seed erzeugt jeder Lauf andere Zufallszahlen →
andere Gewichts-Initialisierung → andere Trainingsergebnisse.

**Konkretes Problem ohne Seed:**
Lauf 1: Accuracy = 74.3%
Lauf 2: Accuracy = 71.8%
Lauf 3: Accuracy = 75.1%
Ist Lauf 2 schlechter weil das Modell schlechter ist —
oder weil die Zufallsinitialisierung ungünstiger war?
Nicht unterscheidbar ohne fixen Seed.

**Mit Seed = 42:**
Lauf 1: Accuracy = 74.3%
Lauf 2: Accuracy = 74.3%
Lauf 3: Accuracy = 74.3%
Ergebnisse sind reproduzierbar — Unterschiede zwischen Experimenten
sind auf Methoden zurückführbar, nicht auf Zufall.

#### Konzept: Warum reicht ein Seed nicht?
TensorFlow, NumPy und Python haben jeweils eigene Zufallsgeneratoren.
Ein `np.random.seed(42)` fixiert nur NumPy — TensorFlow bleibt zufällig.
Alle vier müssen gesetzt werden für vollständige Reproduzierbarkeit.

**Wichtige Einschränkung:**
GPU-Operationen sind inhärent nicht-deterministisch —
parallele Berechnungen auf tausenden GPU-Kernen können
in unterschiedlicher Reihenfolge abgeschlossen werden.
Seed reduziert Varianz, eliminiert sie aber nicht vollständig bei GPU-Training.
Das ist bekannt und akzeptiert in der ML-Community.

#### Ergebnis
Imports erfolgreich.
NumPy     : 2.0.2
TensorFlow: 2.20.0
Seed      : 42
Alle Bibliotheken geladen. Reproduzierbarkeit soweit möglich sichergestellt.

Warum SEED = 42 wichtig ist:
Ohne fixen Seed liefert jedes Training andere Ergebnisse. Kritische Analyse und Vergleich von Experimenten wird sonst unmöglich — Unterschiede könnten Zufall sein, nicht Methodenunterschied.

In [ ]:
# =============================================================================
# Zelle 05 – GitHub Verbindung einrichten (DEAKTIVIERT)
# =============================================================================
# GRUND: Token war hardcoded im Code → Sicherheitsrisiko.
# LÖSUNG: Token wird ab Zelle 06 aus Colab Secrets geladen.
# Diese Zelle bleibt zur Dokumentation des Lernpfads erhalten.
# Token und E-Mail wurden unkenntlich gemacht.
#
# NICHT AUSFÜHREN
# =============================================================================
#
# import os
# from google.colab import userdata
#
# ── Konfiguration ──────────────────────────────────────────────────────────────
# GITHUB_USERNAME = "AwaTekoete"
# GITHUB_EMAIL    = "XXXX@XXXX.com"                  # ← ENTFERNT
# GITHUB_TOKEN    = "ghp_XXXXXXXXXXXXXXXXXXXX"        # ← ENTFERNT
# GITHUB_REPO_URL = f"https://{GITHUB_TOKEN}@github.com/AwaTekoete/MIST_CV_CIFAR10.git"
#
# REPO_LOCAL_PATH = '/content/MIST_CV_CIFAR10'
#
# ── Git Identität setzen ───────────────────────────────────────────────────────
# os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
# os.system(f'git config --global user.email "{GITHUB_EMAIL}"')
#
# ── Repository klonen ─────────────────────────────────────────────────────────
# if not os.path.exists(REPO_LOCAL_PATH):
#     ret = os.system(f'git clone {GITHUB_REPO_URL} {REPO_LOCAL_PATH}')
#     if ret == 0:
#         print(f"✓ Repository geklont nach: {REPO_LOCAL_PATH}")
#     else:
#         print("✗ Klonen fehlgeschlagen – Token oder URL prüfen")
# else:
#     print(f"✓ Repository bereits vorhanden: {REPO_LOCAL_PATH}")
#
# ── Git Status – subprocess für korrekte Verzeichnis-Kontrolle ────────────────
# os.system() öffnet jedes Mal neue Shell → cd gilt nicht für nächsten Befehl
# subprocess mit cwd= setzt Arbeitsverzeichnis korrekt
# import subprocess
#
# result = subprocess.run(
#     ['git', 'status'],
#     cwd=REPO_LOCAL_PATH,          # ← Arbeitsverzeichnis explizit setzen
#     capture_output=True,
#     text=True
# )
#
# print("\nGit Status:")
# print(result.stdout)
#
# if result.returncode != 0:
#     print("Fehler:")
#     print(result.stderr)

✓ Repository bereits vorhanden: /content/MIST_CV_CIFAR10

Git Status:
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean



### Zelle 05 — Original GitHub Setup (DEAKTIVIERT)

#### Was
Ursprüngliche Version der GitHub-Verbindung — deaktiviert und dokumentiert.
Token und E-Mail wurden unkenntlich gemacht.

#### Warum deaktiviert
Der GitHub-Token war hardcoded im Code:
GITHUB_TOKEN = "ghp_XXXXXXXXXXXXXXXXXXXX"
Ein hardcoded Token ist ein kritisches Sicherheitsrisiko:

| Risiko | Konsequenz |
|--------|-----------|
| Token in GitHub committed | Automatische Scanner finden Token in Sekunden |
| Token geteilt (z.B. Screenshot) | Vollzugriff auf GitHub-Account möglich |
| Token in Notebook gespeichert | Jeder mit Notebook-Zugriff hat Token-Zugriff |

GitHub selbst scannt alle Commits automatisch nach Token-Mustern
und deaktiviert gefundene Token sofort — das zeigt wie ernst das Risiko ist.

#### Lernpfad — Warum bleibt die Zelle sichtbar?
Diese Zelle dokumentiert eine bewusste Lernentscheidung:
Fehler sichtbar lassen → Lösung direkt danach → Entwicklung nachvollziehbar.

In professionellen Projekten nennt man das **Architectural Decision Record (ADR)**:
Entscheidungen werden dokumentiert — inklusive was nicht funktioniert hat und warum.

#### Lösung
Token wird ab Zelle 06 aus Colab Secrets geladen —
niemals im Code gespeichert, niemals in GitHub committed.

#### Sicherheitsvorfall — was passiert ist
Der echte Token wurde versehentlich im Chat sichtbar.
Sofortmaßnahme: Token auf GitHub deaktiviert, neuer Token erstellt,
neuer Token in Colab Secrets gespeichert.
Reaktionszeit: < 5 Minuten — das ist der korrekte Prozess.


Sicherheitshinweis: Token niemals direkt in ein Notebook committen das auf GitHub landet. Nach dem Setup wird der Token aus dem Code entfernt und durch eine Umgebungsvariable ersetzt — das kommt in Zelle 06.

Zelle 06 — Token sichern (Sicherheit)
Token darf nie im Notebook-Code stehen der auf GitHub landet. Colab bietet dafür einen sicheren Secret-Speicher.
Einmalig in Colab einrichten:

Colab → linke Seitenleiste → Schlüsselsymbol (🔑 Secrets)
+ Add new secret
Name: GITHUB_TOKEN
Value: Token einfügen
Notebook access aktivieren

In [ ]:
# =============================================================================
# Zelle 06 – Token-Sicherheit + Konfiguration zentralisieren
# =============================================================================
# Token wird aus Colab Secrets geladen – niemals hardcoded im Code.
# Diese Zelle muss bei jeder neuen Session ausgeführt werden.

import subprocess
from google.colab import userdata

# ── Konfiguration zentral ──────────────────────────────────────────────────────
CONFIG = {
    'base_path'    : '/content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10',
    'repo_path'    : '/content/MIST_CV_CIFAR10',
    'github_user'  : 'AwaTekoete',
    'github_email' : 'erik.gerst@hotmail.com',        # ← anpassen
    'seed'         : 42,
}

# ── Token aus Secrets laden ────────────────────────────────────────────────────
try:
    token = userdata.get('GITHUB_TOKEN')
    CONFIG['github_token'] = token

    # Remote URL mit Token aktualisieren (für Push)
    remote_url = f"https://{token}@github.com/AwaTekoete/MIST_CV_CIFAR10.git"
    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', remote_url],
        cwd=CONFIG['repo_path'],
        capture_output=True
    )
    print("✓ Token aus Secrets geladen")
    print("✓ Remote URL aktualisiert")

except Exception as e:
    print(f"✗ Token nicht gefunden: {e}")
    print("→ Colab Secrets prüfen (🔑 linke Seitenleiste)")

# ── Konfiguration ausgeben (Token wird nicht angezeigt) ───────────────────────
print("\nAktive Konfiguration:")
for key, value in CONFIG.items():
    if key == 'github_token':
        print(f"  {key:<15}: ***")
    else:
        print(f"  {key:<15}: {value}")

✓ Token aus Secrets geladen
✓ Remote URL aktualisiert

Aktive Konfiguration:
  base_path      : /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10
  repo_path      : /content/MIST_CV_CIFAR10
  github_user    : AwaTekoete
  github_email   : erik.gerst@hotmail.com
  seed           : 42
  github_token   : ***


### Zelle 06 — Konfiguration + Token aus Secrets

#### Was
Zentrale Projektkonfiguration wird als Python-Dictionary `CONFIG` definiert.
GitHub-Token wird sicher aus Colab Secrets geladen — nicht aus dem Code.
Remote URL wird mit Token aktualisiert damit `git push` funktioniert.

#### Warum — Zentrale Konfiguration
Alle Projektpfade und Parameter an einem Ort:

| Key | Wert | Zweck |
|-----|------|-------|
| `base_path` | Drive-Pfad | Alle Dateien relativ dazu |
| `repo_path` | Lokaler Git-Pfad | Git-Operationen |
| `github_user` | AwaTekoete | Commits, Remote URL |
| `github_email` | erik.gerst@... | Git-Identität |
| `seed` | 42 | Reproduzierbarkeit |
| `github_token` | *** | Authentifizierung |

**Vorteil:** Pfad ändert sich → einmal in CONFIG anpassen →
alle Zellen funktionieren weiterhin. Kein Suchen und Ersetzen im Notebook.

#### Warum — Secrets statt hardcoded Token

Colab Secrets ist ein verschlüsselter Key-Value-Speicher:
- Token wird verschlüsselt gespeichert
- Nur das eigene Google-Konto hat Zugriff
- Token erscheint nie im Notebook-Code
- Token erscheint nie in GitHub

**Vergleich Sicherheitsstufen:**

| Methode | Sicherheit | Empfehlung |
|---------|-----------|------------|
| Hardcoded im Code | ❌ Kritisch | Niemals |
| In Kommentar | ❌ Kritisch | Niemals |
| In `.env` Datei (lokal) | ⚠️ Mittel | Nur lokal, nie committen |
| In Umgebungsvariable | ✅ Gut | Akzeptabel |
| In Colab Secrets | ✅ Gut | Empfohlen für Colab |
| In GitHub Secrets | ✅ Sehr gut | Standard für CI/CD |

#### Konzept: Warum ist Token-Sicherheit so wichtig?
Ein GitHub Personal Access Token mit `repo`-Scope hat vollständigen
Schreibzugriff auf alle Repositories des Accounts.
Ein kompromittierter Token ermöglicht:
- Löschen aller Repositories
- Einschleusen von Schadcode
- Zugriff auf private Repositories

GitHub deaktiviert Token automatisch wenn sie in einem
öffentlichen Repository gefunden werden — Reaktionszeit: Sekunden.

#### Konzept: Warum `CONFIG` als Dictionary?
In größeren Projekten wird `CONFIG` oft aus einer separaten
Konfigurationsdatei geladen (z.B. `config.yaml`).
Dictionary ist der erste Schritt in diese Richtung —
gleiche Denkweise, einfachere Implementierung für dieses Projekt.

#### Wichtig: Session-Verhalten
`CONFIG` existiert nur im RAM der aktuellen Session.
Bei Session-Ende: `CONFIG` verloren.
**Zelle 06 muss bei jeder neuen Session als erstes ausgeführt werden**
(nach Zelle 01 und 04).

#### Ergebnis
✓ Token aus Secrets geladen
✓ Remote URL aktualisiert
Aktive Konfiguration:
base_path      : /content/drive/MyDrive/...
repo_path      : /content/MIST_CV_CIFAR10
github_user    : AwaTekoete
github_email   : erik.gerst@hotmail.com
seed           : 42
github_token   : ***
Konfiguration vollständig. Token sicher geladen.

In [ ]:
# =============================================================================
# Zelle 07 – GitHub Repository klonen
# =============================================================================
# Bereinigte Version von Zelle 05.
# Token wird aus CONFIG geladen – kein hardcoded Token im Code.
# VORAUSSETZUNG: Zelle 06 muss vor dieser Zelle ausgeführt sein.
# =============================================================================

import os
import subprocess

REPO_LOCAL_PATH = CONFIG['repo_path']
GITHUB_REPO_URL = f"https://{CONFIG['github_token']}@github.com/{CONFIG['github_user']}/MIST_CV_CIFAR10.git"

# ── Repository klonen (nur wenn noch nicht vorhanden) ─────────────────────────
if not os.path.exists(REPO_LOCAL_PATH):
    ret = os.system(f'git clone {GITHUB_REPO_URL} {REPO_LOCAL_PATH}')
    if ret == 0:
        print(f"✓ Repository geklont nach: {REPO_LOCAL_PATH}")
    else:
        print("✗ Klonen fehlgeschlagen – Token oder URL prüfen")
else:
    print(f"✓ Repository bereits vorhanden: {REPO_LOCAL_PATH}")

# ── Git Identität setzen ───────────────────────────────────────────────────────
os.system(f'git config --global user.name "{CONFIG["github_user"]}"')
os.system(f'git config --global user.email "{CONFIG["github_email"]}"')

# ── Git Status ────────────────────────────────────────────────────────────────
result = subprocess.run(
    ['git', 'status'],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True
)
print(f"\nGit Status:\n{result.stdout}")

if result.returncode != 0:
    print(f"✗ Fehler: {result.stderr}")

✓ Repository bereits vorhanden: /content/MIST_CV_CIFAR10

Git Status:
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)



### Zelle 07 — GitHub Repository klonen (bereinigt)

#### Was
Bereinigte Version von Zelle 05.
Repository wird von GitHub auf den lokalen Colab-Server geklont.
Token wird aus `CONFIG` geladen — kein hardcoded Token im Code.

#### Warum — Klonen statt neu erstellen
Das Repository existiert bereits auf GitHub (mit README, LICENSE, .gitignore).
`git clone` kopiert den kompletten Versionsstand lokal —
inklusive gesamter Commit-Historie.

**Arbeitsfluss:**
GitHub (Remote) ←→ /content/MIST_CV_CIFAR10 (Lokal) ←→ Drive (Persistent)
git push →                                    shutil.copy →
← git pull                                    ← shutil.copy
#### Warum — `subprocess` statt `os.system()`

Zelle 05 (original) verwendete `os.system()` — das hatte ein Problem:

| Methode | Verhalten | Problem |
|---------|-----------|---------|
| `os.system('cd X && git status')` | Neue Shell pro Aufruf | `cd` gilt nicht für nächsten Befehl |
| `subprocess.run(cwd='X')` | Arbeitsverzeichnis explizit | Korrekt, kontrollierbar |

`subprocess.run()` mit `cwd=` Parameter setzt das Arbeitsverzeichnis
dauerhaft für diesen Aufruf — unabhängig vom aktuellen Verzeichnis.
Zusätzlich: `capture_output=True` fängt stdout und stderr ab →
Fehler können programmatisch ausgewertet werden.

#### Konzept: Git Grundprinzip
Git ist ein verteiltes Versionskontrollsystem:
- Jeder Clone ist eine vollständige Kopie des Repositories
- Commits werden lokal erstellt
- `git push` synchronisiert lokale Commits mit Remote (GitHub)
- `git pull` holt neue Commits von Remote

**Warum Versionskontrolle in ML-Projekten?**

| Ohne Git | Mit Git |
|----------|---------|
| "model_final_v3_FINAL2.keras" | Commit-Hash + Message |
| Änderungen unklar | Diff zeigt exakt was geändert wurde |
| Kein Rollback | Jeder Stand wiederherstellbar |
| Kein Lernfortschritt sichtbar | Commit-Historie = Lernpfad |

#### Konzept: Warum ist das Repo lokal in `/content/` und nicht in Drive?
Git-Operationen (add, commit, push) sind I/O-intensiv.
Drive-Zugriff über Colab-Mount ist langsamer als lokales Dateisystem.
Lösung: Git-Repo lokal in `/content/` → schnelle Git-Operationen.
Notebooks werden per `shutil.copy()` von Drive ins Repo kopiert → dann gepusht.

#### Ergebnis
✓ Repository bereits vorhanden: /content/MIST_CV_CIFAR10
Git Status:
On branch main
Your branch is up to date with 'origin/main'.
nothing to commit, working tree clean
Repository lokal verfügbar. Git-Verbindung zu GitHub aktiv.

In [ ]:
# =============================================================================
# Zelle 08 – Abhängigkeiten installieren + requirements.txt erstellen
# =============================================================================
# Alle benötigten Bibliotheken werden installiert und versioniert dokumentiert.
# requirements.txt ermöglicht reproduzierbare Umgebung auf jedem System.
# =============================================================================

import subprocess
import sys

# ── Bibliotheken installieren ─────────────────────────────────────────────────
packages = [
    'tensorflow',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',    # Metriken: F1, Confusion Matrix, Classification Report
    'Pillow',          # Bildverarbeitung
    'opencv-python',   # Bildqualitäts-Checks (Laplacian Variance etc.)
]

print("Installiere Pakete...")
for package in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', package, '-q'],
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print(f"✓ {package}")
    else:
        print(f"✗ {package} – Fehler: {result.stderr[:100]}")

# ── Installierte Versionen erfassen ───────────────────────────────────────────
import tensorflow as tf
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import PIL
import cv2

versions = {
    'tensorflow'      : tf.__version__,
    'numpy'           : np.__version__,
    'matplotlib'      : matplotlib.__version__,
    'seaborn'         : sns.__version__,
    'scikit-learn'    : sklearn.__version__,
    'Pillow'          : PIL.__version__,
    'opencv-python'   : cv2.__version__,
}

print("\nInstallierte Versionen:")
for lib, ver in versions.items():
    print(f"  {lib:<20}: {ver}")

# ── requirements.txt erstellen ────────────────────────────────────────────────
req_path = os.path.join(CONFIG['base_path'], 'requirements.txt')

with open(req_path, 'w') as f:
    f.write("# MIST CV CIFAR-10 – Requirements\n")
    f.write("# Generiert automatisch – Zelle 08\n\n")
    for lib, ver in versions.items():
        f.write(f"{lib}=={ver}\n")

print(f"\n✓ requirements.txt erstellt: {req_path}")

Installiere Pakete...
✓ tensorflow
✓ numpy
✓ matplotlib
✓ seaborn
✓ scikit-learn
✓ Pillow
✓ opencv-python

Installierte Versionen:
  tensorflow          : 2.20.0
  numpy               : 2.0.2
  matplotlib          : 3.10.0
  seaborn             : 0.13.2
  scikit-learn        : 1.6.1
  Pillow              : 11.3.0
  opencv-python       : 4.13.0

✓ requirements.txt erstellt: /content/drive/MyDrive/_Masterschool/Term_09/ComputerVision/MIST_CV_CIFAR10/requirements.txt


### Zelle 08 — Abhängigkeiten installieren + requirements.txt

#### Was
Alle benötigten Bibliotheken werden installiert und mit exakten
Versionsnummern in `requirements.txt` dokumentiert.

#### Warum — diese Bibliotheken

| Bibliothek | Version | Aufgabe in diesem Projekt |
|------------|---------|--------------------------|
| `tensorflow` | 2.20.0 | Modellbau, Training, Evaluation |
| `numpy` | 2.0.2 | Array-Operationen, Datenmanipulation |
| `matplotlib` | 3.10.0 | Plots, Trainingsverläufe, Visualisierungen |
| `seaborn` | 0.13.2 | Statistische Plots, Confusion Matrix |
| `scikit-learn` | 1.6.1 | Metriken: F1, Classification Report |
| `Pillow` | 11.3.0 | Bildverarbeitung, Bildformate lesen |
| `opencv-python` | 4.13.0 | Bildqualitäts-Checks (Laplacian Variance, SNR) |

#### Warum — scikit-learn für Metriken
TensorFlow liefert während des Trainings nur Accuracy und Loss.
Für professionelle Evaluation braucht man mehr:

| Metrik | Bibliothek | Warum wichtig |
|--------|-----------|---------------|
| Accuracy | TensorFlow | Basismetrik — aber allein unzureichend |
| Macro F1 | scikit-learn | Klassenbalance-fair |
| Per-Class F1 | scikit-learn | Zeigt wo Modell versagt |
| Confusion Matrix | scikit-learn | Visualisiert Verwechslungen |
| Classification Report | scikit-learn | Vollständige Übersicht |

#### Warum — opencv für Bildqualität
OpenCV ermöglicht quantitative Bildqualitäts-Checks:

**Laplacian Variance** — misst Schärfe:
scharf = hohe Varianz der zweiten Ableitung
unscharf = niedrige Varianz
Formel: `cv2.Laplacian(image, cv2.CV_64F).var()`
Schwellwert: < 100 = potenziell unscharf (datensatzabhängig)

**Signal-to-Noise Ratio (SNR):**
SNR = mean(pixel) / std(pixel)
hoher SNR = klares Signal, wenig Rauschen
niedriger SNR = verrauschtes Bild
Diese Metriken werden in Notebook 02 (EDA) eingesetzt um
Bildqualität des CIFAR-10 Datensatzes quantitativ zu bewerten.

#### Konzept: Warum requirements.txt?
`requirements.txt` dokumentiert exakte Bibliotheksversionen:
tensorflow==2.20.0
numpy==2.0.2
...
**Reproduzierbarkeit:** Gleiche Versionen → gleiche Ergebnisse.
Unterschiedliche Versionen können zu unterschiedlichen Ergebnissen führen —
besonders bei TensorFlow zwischen Major-Versionen (1.x vs 2.x).

**Professioneller Standard:**
```bash
# Projekt auf neuem System einrichten:
pip install -r requirements.txt
# → exakt dieselbe Umgebung
```

#### Konzept: Versionspinning — Vor- und Nachteile

| | Vorteil | Nachteil |
|-|---------|---------|
| Exakte Version (`==`) | Vollständig reproduzierbar | Keine Sicherheitsupdates |
| Mindestversion (`>=`) | Immer aktuell | Möglicherweise inkompatibel |
| Keine Angabe | Einfach | Nicht reproduzierbar |

Für ML-Projekte: exakte Versionspinning (`==`) ist Standard.

#### Ergebnis
✓ tensorflow
✓ numpy
✓ matplotlib
✓ seaborn
✓ scikit-learn
✓ Pillow
✓ opencv-python
✓ requirements.txt erstellt
Alle Bibliotheken installiert und versioniert dokumentiert.

In [ ]:
# =============================================================================
# Zelle 09 – Smoke Test
# =============================================================================
# Prüft ob alle Komponenten korrekt eingerichtet sind.
# Smoke Test = minimaler Test ob System grundsätzlich funktioniert.
# Kein inhaltlicher Test – nur Infrastruktur.
# =============================================================================

import os
import subprocess
import tensorflow as tf

print("=" * 55)
print("SMOKE TEST – Setup Verifikation")
print("=" * 55)

tests = []

# ── Test 1: Google Drive ──────────────────────────────────────────────────────
drive_ok = os.path.exists(CONFIG['base_path'])
tests.append(('Google Drive gemountet', drive_ok))

# ── Test 2: Ordnerstruktur ────────────────────────────────────────────────────
expected_folders = [
    'notebooks',
    'models',
    'reports/figures',
    'reports/metrics',
    'reports/misclassified',
    'presentation',
]
folders_ok = all(
    os.path.exists(os.path.join(CONFIG['base_path'], f))
    for f in expected_folders
)
tests.append(('Ordnerstruktur vollständig', folders_ok))

# ── Test 3: GPU ───────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
gpu_ok = len(gpus) > 0
tests.append(('GPU verfügbar', gpu_ok))

# ── Test 4: Seed gesetzt ──────────────────────────────────────────────────────
seed_ok = CONFIG['seed'] == 42
tests.append(('Seed konfiguriert', seed_ok))

# ── Test 5: GitHub Verbindung ─────────────────────────────────────────────────
result = subprocess.run(
    ['git', 'status'],
    cwd=CONFIG['repo_path'],
    capture_output=True,
    text=True
)
github_ok = result.returncode == 0
tests.append(('GitHub Verbindung aktiv', github_ok))

# ── Test 6: requirements.txt vorhanden ───────────────────────────────────────
req_ok = os.path.exists(os.path.join(CONFIG['base_path'], 'requirements.txt'))
tests.append(('requirements.txt vorhanden', req_ok))

# ── Test 7: TensorFlow Basisoperation ────────────────────────────────────────
try:
    x = tf.constant([1.0, 2.0, 3.0])
    y = tf.reduce_mean(x)
    tf_ok = float(y.numpy()) == 2.0
except Exception:
    tf_ok = False
tests.append(('TensorFlow Basisoperation', tf_ok))

# ── Ergebnis ausgeben ─────────────────────────────────────────────────────────
print()
all_passed = True
for test_name, passed in tests:
    status = "✓" if passed else "✗"
    print(f"  {status}  {test_name}")
    if not passed:
        all_passed = False

print()
print("=" * 55)
if all_passed:
    print("ERGEBNIS: Alle Tests bestanden – Setup vollständig.")
else:
    print("ERGEBNIS: Fehler gefunden – Details oben prüfen.")
print("=" * 55)

SMOKE TEST – Setup Verifikation

  ✓  Google Drive gemountet
  ✓  Ordnerstruktur vollständig
  ✓  GPU verfügbar
  ✓  Seed konfiguriert
  ✓  GitHub Verbindung aktiv
  ✓  requirements.txt vorhanden
  ✓  TensorFlow Basisoperation

ERGEBNIS: Alle Tests bestanden – Setup vollständig.


### Zelle 09 — Smoke Test

#### Was
Minimaler Systemtest der prüft ob alle Komponenten korrekt eingerichtet sind.
7 Tests werden sequentiell ausgeführt — bei Fehler ist sofort klar wo das Problem liegt.

#### Warum — Smoke Test als Konzept
Der Begriff kommt aus der Hardware-Entwicklung:
"Gerät einschalten — wenn es raucht, sofort ausschalten."
In Software: minimaler Test ob das System grundsätzlich funktioniert —
kein inhaltlicher Test, nur Infrastruktur.

**Warum vor dem eigentlichen Projekt?**
Ohne Smoke Test können Fehler erst Stunden später sichtbar werden:
Training läuft 2 Stunden →
Speichern fehlgeschlagen (Drive nicht gemountet) →
Alle Ergebnisse verloren
Mit Smoke Test: Problem wird in Sekunden erkannt, nicht nach Stunden.

#### Die 7 Tests — Warum jeder davon wichtig ist

| Test | Was wird geprüft | Konsequenz bei Fehler |
|------|-----------------|----------------------|
| Google Drive | Drive gemountet | Kein persistenter Speicher |
| Ordnerstruktur | Alle Ordner vorhanden | Plots/Modelle können nicht gespeichert werden |
| GPU | T4 verfügbar | Training ~15x langsamer |
| Seed | CONFIG['seed'] == 42 | Reproduzierbarkeit nicht garantiert |
| GitHub | git status erfolgreich | Kein Versionskontrolle möglich |
| requirements.txt | Datei vorhanden | Umgebung nicht dokumentiert |
| TensorFlow | Basisoperation korrekt | Modellbau nicht möglich |

#### Konzept: Idempotenz
Smoke Test kann beliebig oft ausgeführt werden — immer dasselbe Ergebnis.
Kein Test verändert den Systemzustand.
Das ist eine wichtige Eigenschaft von Tests: **idempotent** =
mehrfache Ausführung hat dieselbe Wirkung wie einmalige Ausführung.

#### Konzept: Test-Driven Development (TDD) Grundgedanke
In professioneller Software-Entwicklung werden Tests vor dem Code geschrieben.
Smoke Tests sind der einfachste Einstieg in diese Denkweise:
- Was muss funktionieren?
- Wie messe ich ob es funktioniert?
- Was ist die Konsequenz wenn es nicht funktioniert?

Diese Denkweise wird später bei der Modell-Evaluation wieder aufgegriffen:
- Was soll das Modell können?
- Wie messe ich die Leistung?
- Was bedeutet ein schlechtes Ergebnis?

#### Ergebnis
=======================================================
SMOKE TEST – Setup Verifikation
✓  Google Drive gemountet
✓  Ordnerstruktur vollständig
✓  GPU verfügbar
✓  Seed konfiguriert
✓  GitHub Verbindung aktiv
✓  requirements.txt vorhanden
✓  TensorFlow Basisoperation
=======================================================
ERGEBNIS: Alle Tests bestanden – Setup vollständig.
Alle 7 Tests bestanden. Setup ist vollständig und verifiziert.
Infrastruktur ist bereit für Notebook 02 — EDA.

In [ ]:
# =============================================================================
# Zelle 10 – Ersten Commit auf GitHub pushen
# =============================================================================
# Notebook und requirements.txt werden ins GitHub Repository kopiert
# und als erster Commit gespeichert.
# Konvention: chore: für Setup- und Konfigurations-Commits
# =============================================================================

import shutil
import subprocess
import os

REPO_PATH = CONFIG['repo_path']

# ── Ordnerstruktur im Repo anlegen ────────────────────────────────────────────
os.makedirs(os.path.join(REPO_PATH, 'notebooks'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'figures'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'metrics'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'reports', 'misclassified'), exist_ok=True)
os.makedirs(os.path.join(REPO_PATH, 'presentation'), exist_ok=True)
print("✓ Ordnerstruktur im Repo angelegt")

# ── requirements.txt kopieren ─────────────────────────────────────────────────
src_req = os.path.join(CONFIG['base_path'], 'requirements.txt')
dst_req = os.path.join(REPO_PATH, 'requirements.txt')
shutil.copy(src_req, dst_req)
print("✓ requirements.txt kopiert")

# ── Notebook kopieren ───────────────────────────────

✓ Ordnerstruktur im Repo angelegt
✓ requirements.txt kopiert


### Zelle 10 — Ersten Commit auf GitHub pushen

#### Was
Notebook und requirements.txt werden ins lokale Git-Repository kopiert
und als erster inhaltlicher Commit auf GitHub gepusht.

Drei Git-Operationen in Sequenz:
1. `git add .` — alle Änderungen zum Commit vormerken
2. `git commit` — Snapshot erstellen mit Beschreibung
3. `git push` — Snapshot auf GitHub hochladen

#### Warum — Git Workflow verstehen

**Der fundamentale Git-Kreislauf:**
Arbeitsverzeichnis → git add → Staging Area → git commit →
Lokales Repo → git push → GitHub (Remote)
Jede Stufe hat eine Bedeutung:

| Stufe | Analogie | Zweck |
|-------|---------|-------|
| Arbeitsverzeichnis | Schreibtisch | Aktuelle Arbeit |
| Staging Area (`git add`) | Postkorb | Was kommt in nächsten Commit |
| Lokales Repo (`git commit`) | Archiv | Gespeicherter Snapshot |
| GitHub (`git push`) | Bibliothek | Geteilt, gesichert, versioniert |

#### Warum — Commit-Message Konvention
Commit-Message: `chore: initial project structure and setup notebook`

Format: `type: beschreibung`

| Type | Verwendung | Beispiel |
|------|-----------|---------|
| `chore` | Setup, Konfiguration | `chore: initial project structure` |
| `feat` | Neues Feature / Schritt | `feat: add EDA notebook` |
| `fix` | Bugfix | `fix: correct normalization range` |
| `docs` | Dokumentation | `docs: add README section` |
| `model` | Modell hinzugefügt | `model: add ResNet50 baseline` |
| `viz` | Neue Visualisierung | `viz: add confusion matrix plot` |
| `refactor` | Code umstrukturiert | `refactor: extract config to cell 06` |

**Warum diese Konvention?**
Commit-Historie wird lesbar wie ein Logbuch:
model: add fine-tuning with unfrozen ResNet50
viz: add per-class F1 bar chart
feat: add ModelCheckpoint callback
fix: correct seed initialization order
chore: initial project structure
Jeder Commit ist sofort verständlich — ohne den Code zu öffnen.

#### Konzept: Warum shutil.copy() statt direktem Arbeiten im Repo?
Notebook wird in Colab bearbeitet → automatisch auf Drive gespeichert.
Git-Repo liegt lokal in `/content/` — nicht auf Drive.
`shutil.copy()` überbrückt diese Lücke:
Drive (persistent)     →  shutil.copy()  →  /content/Repo (lokal)
notebooks/               →                   notebooks/
01_setup.ipynb           →                   01_setup.ipynb
↓
git add + commit + push
↓
GitHub (remote)
#### Konzept: Atomare Commits
Jeder Commit sollte eine abgeschlossene, sinnvolle Einheit sein.
Nicht: alles auf einmal am Ende committen.
Sondern: nach jedem Notebook-Abschluss committen.

**Warum?**
- Fehler sind leichter lokalisierbar
- Rollback auf spezifischen Stand möglich
- Lernfortschritt granular sichtbar
- Code-Review wird einfacher

#### Ergebnis
$ git add .
$ git commit -m chore: initial project structure and setup notebook
[main 6ee9e01] chore: initial project structure and setup notebook
2 files changed, 11 insertions(+)
create mode 100644 notebooks/01_setup_environment.ipynb
create mode 100644 requirements.txt
$ git push origin main
✓ Commit und Push erfolgreich
Repository: https://github.com/AwaTekoete/MIST_CV_CIFAR10
2 Dateien committed und auf GitHub gepusht.
Lernfortschritt ist versioniert und öffentlich dokumentiert.

---

## Notebook 01 — Abschluss: Setup Environment

### Durchgeführte Schritte

| Zelle | Inhalt | Status |
|-------|--------|--------|
| 01 | Google Drive mounten | ✅ |
| 02 | Ordnerstruktur anlegen | ✅ |
| 03 | GPU prüfen und dokumentieren | ✅ |
| 04 | Zentrale Imports + Seed | ✅ |
| 05 | Original GitHub Setup (deaktiviert) | ✅ |
| 06 | Konfiguration + Token aus Secrets | ✅ |
| 07 | GitHub Repository klonen (bereinigt) | ✅ |
| 08 | Abhängigkeiten installieren + requirements.txt | ✅ |
| 09 | Smoke Test (7/7 bestanden) | ✅ |
| 10 | Ersten Commit auf GitHub pushen | ✅ |

### Systemkonfiguration

| Parameter | Wert |
|-----------|------|
| TensorFlow | 2.20.0 |
| NumPy | 2.0.2 |
| GPU | T4 (12 GB VRAM) |
| RAM | 13.6 GB |
| Seed | 42 |
| Python | 3.x |

### Gelernte Konzepte

| Konzept | Kernaussage |
|---------|------------|
| Drive Persistenz | Colab ist temporär — Drive ist permanent |
| GPU Notwendigkeit | T4 ist ~15x schneller als CPU für CNN-Training |
| Seed | Reproduzierbarkeit erfordert 4 Zufallsgeneratoren |
| Token-Sicherheit | Niemals hardcoded — immer Secrets oder Umgebungsvariablen |
| Git Workflow | add → commit → push — atomare, beschreibende Commits |
| Smoke Test | Infrastruktur vor Arbeit verifizieren — spart Stunden |
| requirements.txt | Exakte Versionspinning für Reproduzierbarkeit |

### Offene Punkte
- Keine

### Nächster Schritt
**Notebook 02: `02_eda.ipynb`**
Datensatz laden, Qualität messen, Verteilungen analysieren,
Bildqualität quantitativ bewerten, Klassen visualisieren.

### Repository
https://github.com/AwaTekoete/MIST_CV_CIFAR10

---
*Notebook 01 abgeschlossen — Phase 0: Setup vollständig.*